# Fundamentals 00.3 - OpenAI Runtime API

Objetivo: comprobar que la misma API publica ejecuta un agente con `openai-runtime`, sin importar el SDK subyacente. El notebook no importa ni configura directamente el cliente OpenAI.

## Parametros de la demostracion

| Variable | Default | Proposito |
|---|---|---|
| RUN_OPENAI_LIVE | 1 | Usa 0 para desactivar la llamada real. |
| OPENAI_API_KEY | sin valor | Credencial leida por el provider, nunca por el notebook. |
| OPENAI_MODEL | provider default | Override del modelo habilitado para tu proyecto. |

## Contrato de la demostracion

El notebook prueba el provider Unicamente a traves de la fachada publica:

```text
toolkit.runtime - toolkit.system - system.agent - RunResult
```

La celda live se habilita con una variable explicita. Sin credenciales o endpoint, el notebook permanece ejecutable y muestra un skip estructurado. Cuando la variable esta activa, cualquier error real del provider debe ser visible.

In [ ]:
import os

import agentic_systems as toolkit

RUN_OPENAI_LIVE = os.getenv("RUN_OPENAI_LIVE", "1").strip().lower() in {"1", "true", "yes"}
HAS_OPENAI_KEY = bool(os.getenv("OPENAI_API_KEY"))
AGENT_NAME = "openai_public_api_probe"

toolkit.show_json({
    "package": toolkit.__name__,
    "version": toolkit.__version__,
    "run_live": RUN_OPENAI_LIVE,
    "credentials_configured": HAS_OPENAI_KEY,
}, title="Preflight OpenAI")

## 1) Declarar runtime y limites

`runtime.describe()` es diagnostico declarativo: no ejecuta una peticion y no expone el valor de la API key.

In [ ]:
scheduler = toolkit.scheduler(
    timeout_s=60,
    max_retries=1,
    max_tool_calls=1,
    max_turns=3,
    max_concurrency=1,
)

runtime = toolkit.runtime(
    provider="openai-runtime",
    model=os.getenv("OPENAI_MODEL"),
    scheduler=scheduler,
    metadata={"tutorial": "openai-provider-api"},
)

toolkit.show_json(runtime.describe(), title="OpenAI RuntimeConfig")

## 2) Crear system, tool y agent

La funcion local contiene Unicamente logica de dominio. Registro, contrato, policy, ejecucion y resultado pertenecen al toolkit.

In [ ]:
@toolkit.tool
def inspect_public_api(symbol: str) -> dict:
    """Verifica un simbolo contra la superficie publica instalada."""
    return {
        "symbol": symbol,
        "is_public": symbol in toolkit.__all__,
        "package_version": toolkit.__version__,
    }

system = toolkit.system(runtime=runtime)
agent = system.agent(
    name=AGENT_NAME,
    instructions=(
        "Usa inspect_public_api para verificar el simbolo solicitado. "
        "Responde con el nombre, si es publico y la version observada."
    ),
    tools=[inspect_public_api],
    contract=toolkit.AgentContract(
        must_call=["inspect_public_api"],
        completion="when_required_tools_satisfied",
    ),
    policy=toolkit.RunPolicy(
        max_turns=3,
        max_tool_calls=1,
        temperature=0.0,
        trace="compact",
        strict=True,
    ),
)

toolkit.show_json(agent.info(), title="Agente declarado")

La ejecucion live esta habilitada por defecto cuando existe OPENAI_API_KEY. Usa RUN_OPENAI_LIVE=0 para forzar un skip seguro.

In [ ]:
can_run = RUN_OPENAI_LIVE and HAS_OPENAI_KEY

if can_run:
    result = agent.run(
        "Verifica si system pertenece a la API publica instalada.",
        mode="eval",
    )
    toolkit.human_result(result, title="OpenAI RunResult", show_lineage=True)
    toolkit.show_json(toolkit.run_result_output(result), title="Contrato normalizado")
else:
    result = None
    toolkit.show_json({
        "status": "skipped",
        "provider": "openai-runtime",
        "reason": "Configura OPENAI_API_KEY o usa RUN_OPENAI_LIVE=0 para mantener el skip.",
    }, title="OpenAI live gate")

## 4) API realmente ejercitada

La cobertura enumera solamente llamadas presentes en las celdas anteriores.

In [ ]:
api_coverage = [
    "toolkit.scheduler",
    "toolkit.runtime",
    "toolkit.tool",
    "toolkit.system",
    "system.agent",
    "agent.run",
    "toolkit.human_result",
    "toolkit.run_result_output",
    "toolkit.show_json",
    "toolkit.AgentContract",
    "toolkit.RunPolicy",
]

toolkit.show_json(api_coverage, title="OpenAI API coverage")

## Resultado esperado

Con live desactivado: configuracion observable y skip explicito. Con live activado: un `RunResult` real cuyo runtime debe reportar `openai-runtime` y cuya evidencia incluye `inspect_public_api`.